# S7.2 - Playing with Prolog: Apple 2025 Finance Edition

This notebook preserves the original content, code, and visual explanations from `Section 3.2 - Playing with Prolog.ipynb`.

The finance-edition cells at the top connect the same NLP concept to **Apple Inc.'s fiscal 2025 Form 10-K**. The original notebook follows after these bridge cells so students can compare the general NLP pipeline with a realistic financial-document workflow.


In [ ]:
from pathlib import Path
import os
import sys

FINANCE_DIR = Path(r"D:\Repositories\AD698-generative-ai-for-BA\help-code\4-nlp-finance")
FINANCE_SRC = Path(r"D:\Repositories\AD698-generative-ai-for-BA\help-code\4-nlp-finance\src")
ORIGINAL_NOTEBOOK_DIR = Path(r"D:\Repositories\AD698-generative-ai-for-BA\help-code\2-core-tasks")

if str(FINANCE_SRC) not in sys.path:
    sys.path.insert(0, str(FINANCE_SRC))
if str(ORIGINAL_NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(ORIGINAL_NOTEBOOK_DIR))

from finance_nlp_utils import *

finance_doc = load_apple_10k()
finance_sample = sample_sentences(5)
finance_metrics = load_metrics()

print(f"Apple 2025 10-K excerpt characters: {len(finance_doc):,}")
print("Example Apple sentence:")
print(finance_sample[0])

# Run original notebook code from its original folder so relative data/image paths still work.
os.chdir(ORIGINAL_NOTEBOOK_DIR)


In [ ]:
print("Available Apple SEC 10-K evaluation sets")
for name in list_evaluation_sets():
    print("-", name)

print("\nSentence classification example")
classification_rows = load_evaluation_csv("apple_2025_sentence_classification.csv")
for row in classification_rows[:3]:
    print(row["split"], row["label"], "=>", row["sentence"][:120])

print("\nExtractive QA example")
qa_rows = load_evaluation_jsonl("apple_2025_qa.jsonl")
print(qa_rows[0])


<img src='data/images/section-notebook-header.png' />

# Playing with Prolog

## Introduction

### What is First-Order-Logic?

First-order logic, also known as first-order predicate calculus or first-order logic with equality, is a formal system used in mathematical logic and computer science to reason about relationships and properties of objects within a domain. It is a fundamental framework for expressing and analyzing logical statements and is widely used in various areas of computer science and artificial intelligence.

In first-order logic, the domain of discourse consists of objects, and logical statements are expressed in terms of predicates, variables, quantifiers, and logical connectives. Here are some key components of first-order logic:

* **Predicates:** Predicates are used to express relationships or properties that hold between objects. They can be thought of as functions that take objects as arguments and return a truth value. For example, "IsRed(x)" could be a predicate that evaluates to true if object x is red.

* **Variables:** Variables are placeholders that can take on different values from the domain of discourse. They are used to generalize statements and make them applicable to different objects. For example, "IsRed(x)" can be interpreted as "x is red," where x is a variable that can represent any object.

* **Quantifiers:** Quantifiers are used to express the scope of variables and make assertions about the entire domain or subsets of it. The two main quantifiers in first-order logic are the universal quantifier (∀) and the existential quantifier (∃). The universal quantifier (∀) asserts that a statement holds for all objects in the domain, while the existential quantifier (∃) asserts that there exists at least one object for which the statement holds.

* **Logical Connectives:** First-order logic includes the usual logical connectives, such as conjunction (AND), disjunction (OR), negation (NOT), implication (IF-THEN), and equivalence (IF AND ONLY IF). These connectives are used to combine and manipulate logical statements.

Using these components, first-order logic allows you to express complex statements and reason about their truth or falsehood. The formal rules of first-order logic provide a basis for rigorous inference and deduction. First-order logic serves as the foundation for many formal reasoning systems, theorem provers, and programming languages such as **Prolog**. It provides a powerful and expressive language for representing knowledge and making logical inferences about the world.

### What is Prolog?

To make actual use of logical propositions in a programmatically manner, we need logical programming language or other logic engines for formulating and evaluating logic expressions. This is a principle challenge since First-Order Logic is [undecidable](https://en.wikipedia.org/wiki/Decidability_(logic)). As such, many logic programming languages only support a subset of First-Order Logic. In this notebook, we will be using [Prolog](https://en.wikipedia.org/wiki/Prolog), one of the most prominent logic programming languages. Prolog does not support arbitrary first-order logic but only a fragment of it known as [Horn clauses](https://en.wikipedia.org/wiki/Horn_clause). Not every statement in logic can be converted to this form. Prolog is interactive; you load a KB and then ask queries. Prolog adopts the closed-world assumption: (a) all knowledge of the world is present in the database, and (b) If a term is not in the database assume is false.

Prolog is a declarative programming language that is based on the concept of logic programming. It stands for "Programming in Logic" and was developed in the 1970s by Alain Colmerauer and Philippe Roussel at the University of Aix-Marseille in France. Prolog is primarily used for symbolic and logical programming. It is **based on first-order logic** and provides a way to define facts and rules that represent relationships and logical constraints. Programs in Prolog consist of a collection of facts and rules, which are used to make logical inferences and answer queries.

One of the key features of Prolog is its ability to perform automatic backtracking and pattern matching, which allows the programmer to specify what needs to be achieved rather than how to achieve it. This makes Prolog well-suited for applications such as expert systems, natural language processing, automated reasoning, and symbolic computation.

Prolog programs are executed by a Prolog interpreter or compiler, which uses a process called "resolution" to search for solutions to queries based on the defined facts and rules. The interpreter uses a depth-first search strategy to explore possible solutions and backtracks when necessary. Prolog has influenced the development of other programming languages and has been used as the basis for various extensions and implementations. Several Prolog dialects and systems exist, such as **SWI-Prolog**, GNU Prolog, and SICStus Prolog, each with their own specific features and optimizations.

While first-order logic statements provide a general framework for expressing logical relationships, Prolog uses **Horn clauses** which are a simplified form of logical statements (i.e., a subset of statements of first-order logic). First-order logic is more expressive and flexible, allowing for a wider range of logical expressions, while Horn clauses provide a focused representation for logic programming and inference.


### What are Horn Clauses?

Horn clauses are a specific form of logical statement used in Prolog and other logic programming languages. They play a fundamental role in the inference and resolution process of Prolog programs. A Horn clause is an implication with a head and a body, written in the form:

```
Head :- Body.
```

The head represents a single goal or predicate, while the body consists of a conjunction of goals or predicates. Here are a few key characteristics of Horn clauses in Prolog:

* **Head:** The head of a Horn clause represents the goal or conclusion that we want to prove or derive. It is typically a single predicate.

* **Body:** The body of a Horn clause consists of a conjunction of goals or predicates. It represents the conditions or subgoals that need to be satisfied in order to prove the head. The body can be empty, in which case the clause is called a fact.

* **Horn Clause Types:**

	* **Facts:** Horn clauses without a body are considered facts. They represent simple assertions or base cases in Prolog. For example: parent(john, mary).
    
	* **Rules:** Horn clauses with a non-empty body are considered rules. They represent logical implications or relationships. For example: ancestor(X, Y) :- parent(X, Y), ancestor(X, Z). This rule states that if X is a parent of Y and X is also an ancestor of Z, then X is an ancestor of Y.

* **Horn Clause Logic:** The logical interpretation of a Horn clause is that if all the goals in the body are true, then the head can be inferred or derived as true. Prolog uses this logical interpretation to perform backtracking and resolution, attempting to satisfy the goals in the body to prove the head.

Prolog programs consist of a collection of Horn clauses, which are used to define relationships, facts, and rules. The Prolog interpreter or compiler uses these clauses to perform unification, pattern matching, and backtracking to find solutions to queries based on the defined rules and facts.

Horn clauses provide a declarative and logical way to represent knowledge and relationships in Prolog, enabling powerful inference and reasoning capabilities within logic programming systems.


### What is SWI Prolog and PySwip

[SWI Prolog](https://www.swi-prolog.org/) is a popular and widely used implementation of the Prolog programming language. It is an open-source Prolog system that provides a comprehensive development environment for Prolog programming and offers a range of features and libraries. SWI-Prolog has a large and active community of users and contributors, who provide support, documentation, and share libraries and resources. It is widely used in academia and industry for applications such as expert systems, natural language processing, constraint solving, and symbolic computation. Overall, SWI-Prolog offers a powerful and user-friendly environment for Prolog programming, making it an excellent choice for both beginners and experienced Prolog developers.

The [`PySwip`](https://github.com/yuce/pyswip) package is a Python interface to SWI-Prolog, which allows Python programmers to interact with SWI-Prolog from within their Python code. It enables seamless integration between Python and Prolog, enabling you to use Prolog's logic programming capabilities and SWI-Prolog's features within your Python applications. `pyswip` provides a convenient and seamless way to combine the power of Prolog's logic programming with the flexibility and extensibility of Python. It is particularly useful when you want to leverage Prolog's rule-based reasoning, backtracking, and pattern matching capabilities to solve complex problems within a Python application. It's worth noting that `pyswip` is specifically designed to work with SWI-Prolog and may not be compatible with other Prolog implementations.

## Setting up the Notebook

### Import Required Packages

SWI-Prolog offers a command line interface (CLI) for user interaction. However, here we want to use Prolog within Python. [PySwip](https://pypi.org/project/pyswip/) is a Python/SWI-Prolog bridge enabling you to query SWI-Prolog in your Python programs. It features an (incomplete) SWI-Prolog foreign language interface, a utility class that makes it easy querying with Prolog and also a Pythonic interface.

In [ ]:
try:
    from pyswip import Prolog
except Exception as exc:
    print('pyswip/SWI-Prolog is optional and not available:', exc)
    class Prolog:
        def consult(self, *_args, **_kwargs):
            print('Using empty Prolog fallback; install SWI-Prolog + pyswip for real query results.')
        def query(self, _query):
            return []

### Loading the Knowledge Base

In principle, you can add facts and rules programmatically, but for our purpose the easiest way is to put all facts and rules into a file and then load this file. The file we use in this notebook adopts the "Harry Potter" knowledge base we used throughout the lectures, only extended by more facts and rules. You can check and edit the file `data/harry-potter.pl` by adding, removing or changing facts and rules as you like.

In [ ]:
prolog = Prolog()

prolog.consult("data/knowledge-bases/harry-potter.pl")

---

## Simple Queries

In the context of this notebook, simple queries refer to queries to refer to only a single predicate. This also means that there is no need for logical connectives.

### Querying Facts

The simplest queries are the ones checking if a fact is true or false. In other words these queries do not contain any variables. Note that the method `query()` always returns a list of dictionaries where each dictionary is a binding of variables to a solution (see examples further down below). This means that if a query does not contain a variable the dictionary will be empty (assuming the query evaluates to true).

If the query is not true, `query()` will return an empty list. Try the examples below! We know that $\text{human}(harry)$ is a fact in our knowledge base, so the output of the code cell will be an empty dictionary `{}`. In contrast, $\text{human}(dobby)$ is not a fact in our knowledge base, so the list of solutions will be empty (so the code cell below will yield no output at all).

Let's first query the truth value of properties, i.e., predicates with an arity of 1.

In [ ]:
q = "human(harry)"
#q = "human(dobby)"

for solution in prolog.query(q):
    print(solution)

We can also query the truth values for relations, i.e., predicates with an arity larger than 1.

In [ ]:
q = "hates(snape,harry)"
#q = "hates(dumbledore,harry)"

for solution in prolog.query(q):
    print(solution)

For the query `hates(snape,harry)` the output will be an empty dictionary `{}` meaning that the query is true which matches our observation from the knowledge base that "snape" does hate "harry". Since "dumbledore" does not hate "harry" according to the knowledge base, the code cell above will yield no output.

### Querying Solutions

Now let's extend this basic idea to using variables. Here, Prolog tries to bind the variables to all possible solutions. Let's first try predicates with one argument (i.e., properties). For example, we can ask who are all the humans or house elves in our knowledge base.

In [ ]:
q = "human(X)" # Who are all the humans in our KB
#q = "houseelf(Y)" # Who are all the house elves in our KB

for solution in prolog.query(q):
    print(solution)

Of course, the variable name doesn't matter; it's only important to remember that variables are capitalized, everything else is not.

For predicates with 2 or more arguments (i.e., relations) the idea is the same, we just use more variables. In the following, we are looking for all the information about "thing" loving a "thing".

In [ ]:
q = "loves(X,Y)"

for solution in prolog.query(q):
    print(solution)

Not all arguments have to be variables in case of relations. For example, with the query below can find all the information who/what "ron" loves.

In [ ]:
q = "loves(ron,Y)"

for solution in prolog.query(q):
    print(solution)

Our knowledge base assumes that the predicate $\text{loves}$ has an "order". For example $\text{loves(snape,harry)}$ only means the "snape" hates "harry" but not vice versa. If we also would want to say that "harry" hates "snape" we would need to add another fact of the form $\text{loves(snape,harry)}$. This is perfectly fine since a love or hate relationship is not necessarily symmetric.

However, how could we model that a relationship is symmetric but also avoid writing all required facts? Let's assume that a relationship $\text{fight}$ is symmetric, meaning if A fights B and B fights A -- one can argue that this might not always be the case, but that's just our assumption here. To reflect this, a common solution is to introduce an "asymmetric" relation (here $\text{fights_with}$), and then define the symmetric relation as a rule that combines both "directions" with a logical OR. This brings us to the rule in the knowledge base

$$\text{fights}(X,Y)\ \text{ :- }\ \text{fights_with}(X,Y)\ ;\ \text{fights_with}(Y,X).$$

In Prolog, the logical OR is expressed with a semicolon. On a side note, we could formulate this also using two separate rules:

$$\text{fights}(X,Y)\ \text{ :- }\ \text{fights_with}(X,Y).$$
$$\text{fights}(X,Y)\ \text{ :- }\ \text{fights_with}(Y,X).$$

In the following code cell, we query for everyone who was in a fight. Note that we get 2 solutions although we have only the 1 fact $\text{fights_with}(\text{harry},\text{voldemort}).$. This is because we define $\text{fights}$ as a symmetric relation.

In [ ]:
q = "fights(X,Y)"

for solution in prolog.query(q):
    print(solution)

Again, we can use constants instead of using only variables to limit the set of solutions. Below, we want to find everyone who fought with "harry", no matter if "harry" is used as the first or second argument of relation $\text{fights}$.

In [ ]:
q = "fights(X,harry)"
#q = "fights(harry,X)"

for solution in prolog.query(q):
    print(solution)

There's also the notion of anonymous variables, in case we do not care what an argument is bound to. For example, if we just want to know who was in a fight -- independent with whom this fight was -- we can formulate the query as follows:

In [ ]:
q = "fights(_,X)"
#q = "fights(X,_)"

for solution in prolog.query(q):
    print(solution)

Analogously, we ask for truth statements to check if a specific person has ever been in a fight.

In [ ]:
q = "fights(_,harry)"
#q = "fights(harry,_)"

for solution in prolog.query(q):
    print(solution)

Recall that the return value will be an empty dictionary if the result is true, and an empty list if the result is false.

In [ ]:
q = "fights(_,ron)"
#q = "fights(ron,_)"

for solution in prolog.query(q):
    print(solution)

---

## More Complex Queries

More complex queries simply refer to queries that combine multiple propositions using logical connectives. In the example below, we are looking for all $X$ that are human and work at "hogwarts". In Prolog, the logical AND is expressed using a comma `,` (compared to the semicolon `;` for the logical OR).

In [ ]:
q = "works_at(X, hogwarts), human(X)"

for solution in prolog.query(q):
    print(solution)

Prolog will enumerate all bindings of X which satisfy each call -- here 3 solutions -- or fail trying. If you are just interested if there exists ($\exists$) a solution you simply need to check if the returned list is not empty. In this case, we can actually improve the query in such a way that we do not have to find all solutions, since only one will be enough. We can stop finding more solutions after the first binding as follows:

In [ ]:
q = "works_at(X, hogwarts), human(X), !"

for solution in prolog.query(q):
    print(solution)

Without going into too much detail, the `!` operator will stop the evaluation once the parts before the operator have been satisfied. In practice, it can have a significant performance benefit when dealing with huge knowledge bases. To sum up, the exists operator ($\exists$) is implicitly supported in Prolog.

Let's try something different. Assume, we want to find all humans who have **not** been in a fight. While Prolog supports negation with the `\+` operator it's not the same as the logical negation operator. `\+` checks if the proposition can be proven to be true or not. Also, since we don't care about the opponent in the fight, we again make use of an anonymous variable:

In [ ]:
#q = "human(X), \+(fights(X, _))"
q = "human(X), \+(fights(_, X))"

for solution in prolog.query(q):
    print(solution)

With a more elaborate and comprehensive knowledge base, naturally much more interesting questions/queries would be possible. However, here, it's all about getting the basic idea, purpose, and benefits of having a formal representation of statements derived from natural language.

Lastly, let's go back to the closed-world assumption which states that anything that cannot be proven using the existing knowledge base is false. For example, in our "Harry Potter" knowledge base you find the rule that any human that has a wand is a wizard, expressed using the clause `wizard(X) :- human(X), wand(Y), owns(X, Y).` Well, let's try to find all wizards:

In [ ]:
q = "wizard(X)"

for solution in prolog.query(q):
    print(solution)

You can see that "dumbledore" or "snape" are not part of the result. This is because our knowledge base does not contain any information -- fact or rule -- about "dumbledore" or "snape" having a wand. So the test if, say, "dumbledore" is a wizard will evaluate to false. Note that there are different ways to add the information that "dumbledore" is a wizard to the knowledge base:

* Add clause `wizard(dumbledore)` *or*

* Add clauses `wand(dumbledore_wand)` and `owns(dumbledore, dumbledore_wand)` *or*

* Add some rule(s) that would make "dumbledore" a wizard; for example, we could express that all humans working at Hogwarts a wizards using the Horn clause `wizard(X) :- human(X), works_at(X, hogwarts)`.

Of course, we can add all additional approaches listed above to the knowledge base.

---

## Creating the Knowledge Base & Queries

In all the examples in this notebook the knowledge base and all the queries were already logical propositions. In practice, the main challenge in the context of semantically representing statements or sentences is to convert text into logical propositions. For example, given the sentence "Harry has fought with Voldermort", we want to automatically derive a fact such as $\text{fights}(\text{Luke}, \text{Vader})$. Or if we have the sentence "Some humans work at Hogwarts.", we want to get a rule such as $\exists x.(\text{jedi}(X) \wedge \text{Human}(X))$ -- or as a Horn clause in Prolog: `works_at(X, hogwarts), human(X), !`

In the lecture, we touched upon this task by introducing the $\lambda$-calculus and semantic attachments as a systematic way to compute facts and rules from a statement or sentence. However, we also saw that this is a very non-trivial task for anything beyond very basic statements. It is therefore beyond the scope of this notebook. Here, the goal was to get a basic sense how a semantic representation of statements and sentences using First-Order Logic can be used to automatically infer information from a knowledge base.

---

## Summary

First-Order Logic (FOL) can be used in Natural Language Processing (NLP) to represent and reason about knowledge, perform inference, and extract meaning from natural language text. Here are a few ways FOL is applied in NLP:

* **Semantic Representation:** FOL can be used as a formal language for representing the semantics of natural language expressions. By mapping natural language sentences into FOL formulas, we can capture the meaning and logical structure of the sentences. This facilitates automated reasoning and inference over the represented knowledge.

* **Question Answering:** FOL can be used to model the knowledge and reasoning required for question answering systems. By representing facts, rules, and logical relationships in FOL, a question answering system can use logical inference to derive answers to user queries. FOL-based representations can help capture complex relationships and perform deductive reasoning to provide accurate answers.

* **Knowledge Extraction:** FOL can be employed to extract structured knowledge from unstructured text. By applying natural language processing techniques such as named entity recognition, relation extraction, and semantic parsing, FOL representations can be constructed from textual data. These FOL representations can then be used for knowledge representation, inference, and integration with other knowledge bases.

* **Textual Entailment:** FOL can be utilized in textual entailment tasks, where the goal is to determine whether one sentence logically entails another. By representing the meaning of two sentences in FOL and comparing their logical relationships, it becomes possible to assess the entailment relation between them. FOL provides a formal framework for modeling and analyzing the logical relationships between sentences.

* **Natural Language Generation:** FOL can be used in natural language generation tasks, where the goal is to automatically generate coherent and logical text. By starting with FOL representations of facts and rules, natural language generation systems can use templates, grammar rules, and linguistic knowledge to generate human-readable text that is consistent with the underlying logic.

These are just a few examples of how First-Order Logic can be applied in NLP. FOL provides a formal and expressive framework for representing and reasoning about knowledge, enabling more sophisticated language understanding and automated inference capabilities in natural language processing systems.


To make actual use of logical propositions in a programmatically manner, we need logical programming language or other logic engines for formulating and evaluating logic expressions. Logic programming languages, such as Prolog, serve several purposes and have unique characteristics that make them suitable for specific applications. Here are some key purposes and benefits of logic programming languages:

* **Symbolic Reasoning:** Logic programming languages excel at symbolic reasoning and logical inference. They provide a declarative approach where programs express relationships, constraints, and rules rather than specifying the procedural steps for solving a problem. This makes logic programming well-suited for domains that require symbolic manipulation, automated reasoning, and rule-based decision-making.

* **Knowledge Representation:** Logic programming languages offer a powerful means of representing and organizing knowledge. They allow developers to express facts, rules, and relationships in a natural and intuitive way. This makes logic programming useful for building expert systems, knowledge-based applications, and intelligent agents that can reason with complex knowledge bases.

* **Non-Deterministic Computation:** Logic programming languages provide a built-in mechanism for non-deterministic computation and backtracking. This allows them to explore multiple possible solutions to a problem and find all valid solutions through systematic search. Backtracking enables logic programming languages to handle uncertainty, ambiguity, and partial information effectively.

* **Pattern Matching:** Pattern matching is a fundamental operation in logic programming languages. They excel at performing pattern matching between the queries and the defined facts and rules. This makes them well-suited for tasks like information retrieval, database querying, and natural language processing, where pattern matching and unification play a crucial role.

* **Natural Language Processing:** Logic programming languages, such as Prolog, have been widely used in natural language processing (NLP) applications. Their ability to handle symbolic representations, perform pattern matching, and support logical inference makes them useful for tasks like text parsing, semantic analysis, question answering, and dialogue systems.

* **Prototyping and Rapid Development:** Logic programming languages often have a concise and expressive syntax, which makes them well-suited for rapid prototyping and exploratory programming. They enable developers to quickly define rules, facts, and relationships, allowing for agile development and experimentation.

Overall, logic programming languages like Prolog serve the purpose of providing a high-level, declarative, and symbolic approach to problem-solving, knowledge representation, and reasoning. They excel in domains that require symbolic manipulation, logical inference, pattern matching, and non-deterministic computation.